In [0]:
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz


In [0]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

#Leitura dos Dados


In [0]:
path = "/Volumes/workspace/hackthon/hackthon_2025/silver/recarga/recarga/"

In [0]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import pytz

data_exec_inicial = 202410

#converte para datatime
data_dt = datetime.strptime(str(data_exec_inicial), '%Y%m')

#Subtrair por 12 meses
data_exec_final = int((data_dt + relativedelta(months=-12)).strftime('%Y%m'))

print(f"A data inicial é {data_exec_inicial} e a data final é {data_exec_final}")

In [0]:
# Importando o Banco de Dados

df_book_recarga = spark.read.format("delta").load(path)

df_book_recarga.createOrReplaceTempView("df_book_recarga")
df_book_recarga.count()

In [0]:
#Verificando a tabela

display(df_book_recarga,10)

In [0]:
df_book_recarga_01 = spark.sql(f"""
SELECT DISTINCT
NUM_CPF
FROM
  df_book_recarga
WHERE
  SAFRA = '{data_exec_inicial}'
""")

df_book_recarga_01.createOrReplaceTempView("df_book_recarga_01")
df_book_recarga_01.count()

In [0]:
display(df_book_recarga_01,10)

In [0]:
df_book_recarga_02 = spark.sql(f"""
SELECT DISTINCT
  *
FROM 
  df_book_recarga
WHERE
-- cuidado data inicial será sempre maior que a data final (inicial - 12)
  SAFRA >= '{data_exec_final}'
  AND SAFRA <= '{data_exec_inicial}'
""")

df_book_recarga_02.createOrReplaceTempView("df_book_recarga_02")
df_book_recarga_02.count()

In [0]:
display(df_book_recarga_02,10)

In [0]:
df_book_recarga_02.createOrReplaceTempView("df_transacoes")

df_temp_01 = spark.sql("""
-- Criando Tabela Temporaria --
WITH base AS (
    SELECT
        *,
        TO_DATE(CONCAT(SAFRA, '01'), 'yyyyMMdd') AS data_dt
    FROM df_transacoes
)
-- Criação das colunas de janela temporal U1M, U3M, U6M, U9M e U12M
-- Para cada CPF, é identificada a safra mais recente disponível no histórico
-- Em seguida, cada registro é avaliado para verificar se sua data pertence
-- a uma janela móvel de N meses anteriores (inclusive) em relação a essa safra
-- Caso o registro esteja dentro da janela definida, a flag recebe valor 1;
-- caso contrário, recebe valor 0

SELECT
    *,
    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY NUM_CPF), -1)
             AND MAX(data_dt) OVER (PARTITION BY NUM_CPF)
        THEN 1 ELSE 0
    END AS U1M,

    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY NUM_CPF), -3)
             AND MAX(data_dt) OVER (PARTITION BY NUM_CPF)
        THEN 1 ELSE 0
    END AS U3M,


    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY NUM_CPF), -6)
             AND MAX(data_dt) OVER (PARTITION BY NUM_CPF)
        THEN 1 ELSE 0
    END AS U6M,

    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY NUM_CPF), -9)
             AND MAX(data_dt) OVER (PARTITION BY NUM_CPF)
        THEN 1 ELSE 0
    END AS U9M,    

    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY NUM_CPF), -12)
             AND MAX(data_dt) OVER (PARTITION BY NUM_CPF)
        THEN 1 ELSE 0
    END AS U12M

FROM base
ORDER BY NUM_CPF, SAFRA
""")

df_temp_01.createOrReplaceTempView("df_temp_01")
df_temp_01.count()

In [0]:
def contagem_percentual(coluna: str):
    query = f"""
        WITH total AS (
            SELECT COUNT(*) AS total_registros
            FROM df_temp_01
        )
        SELECT
            r.{coluna}                               AS valor_coluna,
            COUNT(*)                                AS qtd_registros,
            ROUND(
                COUNT(*) * 100.0 / t.total_registros,
                2
            )                                        AS pct_registros
        FROM df_temp_01 r
        CROSS JOIN total t
        GROUP BY r.{coluna}, t.total_registros
        ORDER BY qtd_registros DESC
    """
    return spark.sql(query)

In [0]:
print('lista de colunas para tipar')

for col_name in spark.table("df_temp_01").columns:
    if col_name.startswith("COD_"):
        print(f"{col_name}")

In [0]:
df_cod = contagem_percentual("COD_TIPO_CREDITO")
display(df_cod)

In [0]:
print('lista de colunas para tipar')

for col_name in spark.table("df_temp_01").columns:
    if col_name.startswith("DW_"):
        print(f"{col_name}")

In [0]:
print('lista de colunas para filtrar')

for col_name in spark.table("df_temp_01").columns:
    if col_name.startswith("IND_"):
        print(f"{col_name}")

In [0]:
df_resultado = contagem_percentual("IND_METODO_PAGAMENTO")
display(df_resultado)

In [0]:
print('lista de valores')

for col_name in spark.table("df_temp_01").columns:
    if col_name.startswith("VAL_"):
        print(f"{col_name}")

## Book Financeiro

In [0]:
# Dataset val_credito_inserido

df_temp_02 = spark.sql("""

SELECT
    NUM_CPF,
    -- Para 1 mês 
    round(sum(case when U1M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_SUM_U1M_VAL_CREDITO_INSERIDO,
    round(count(case when U1M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_COUNT_U1M_VAL_CREDITO_INSERIDO,
    round(avg(case when U1M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_MED_U1M_VAL_CREDITO_INSERIDO,
    
    -- Para 3 meses
    round(avg(case when U3M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_SUM_U3M_VAL_CREDITO_INSERIDO,
    round(count(case when U3M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_COUNT_U3M_VAL_CREDITO_INSERIDO,
    round(avg(case when U3M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_MED_U3M_VAL_CREDITO_INSERIDO,
    
    -- Para 6 meses
    round(avg(case when U6M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_SUM_U6M_VAL_CREDITO_INSERIDO,
    round(count(case when U6M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_COUNT_U6M_VAL_CREDITO_INSERIDO,
    round(avg(case when U6M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_MED_U6M_VAL_CREDITO_INSERIDO,

    -- Para 9 meses
    round(avg(case when U9M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_SUM_U9M_VAL_CREDITO_INSERIDO,
    round(count(case when U9M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_COUNT_U9M_VAL_CREDITO_INSERIDO,
    round(avg(case when U9M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_MED_U9M_VAL_CREDITO_INSERIDO,

    -- Para 12 meses
    round(avg(case when U12M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_SUM_U12M_VAL_CREDITO_INSERIDO,
    round(count(case when U12M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_COUNT_U12M_VAL_CREDITO_INSERIDO,
    round(avg(case when U12M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_MED_U12M_VAL_CREDITO_INSERIDO   
    
    FROM df_temp_01
GROUP BY NUM_CPF
ORDER BY NUM_CPF
""")

df_temp_02.createOrReplaceTempView("df_temp_02")
df_temp_02.count()

In [0]:
display(df_temp_02.limit(10))

In [0]:
# Dataset val_bonus

df_temp_03 = spark.sql("""

SELECT
    NUM_CPF,
    -- Para 1 mês 
    round(sum(case when U1M = 1 then VAL_BONUS else NULL end),2) as VL_SUM_U1M_VAL_BONUS,
    round(count(case when U1M = 1 then VAL_BONUS else NULL end),2) as VL_COUNT_U1M_VAL_BONUS,
    round(avg(case when U1M = 1 then VAL_BONUS else NULL end),2) as VL_MED_U1M_VAL_BONUS,
    
    -- Para 3 meses
    round(avg(case when U3M = 1 then VAL_BONUS else NULL end),2) as VL_SUM_U3M_VAL_BONUS,
    round(count(case when U3M = 1 then VAL_BONUS else NULL end),2) as VL_COUNT_U3M_VAL_BONUS,
    round(avg(case when U3M = 1 then VAL_BONUS else NULL end),2) as VL_MED_U3M_VAL_BONUS,
    
    -- Para 6 meses
    round(avg(case when U6M = 1 then VAL_BONUS else NULL end),2) as VL_SUM_U6M_VAL_BONUS,
    round(count(case when U6M = 1 then VAL_BONUS else NULL end),2) as VL_COUNT_U6M_VAL_BONUS,
    round(avg(case when U6M = 1 then VAL_BONUS else NULL end),2) as VL_MED_U6M_VAL_BONUS,

    -- Para 9 meses
    round(avg(case when U9M = 1 then VAL_BONUS else NULL end),2) as VL_SUM_U9M_VAL_BONUS,
    round(count(case when U9M = 1 then VAL_BONUS else NULL end),2) as VL_COUNT_U9M_VAL_BONUS,
    round(avg(case when U9M = 1 then VAL_BONUS else NULL end),2) as VL_MED_U9M_VAL_BONUS,

    -- Para 12 meses
    round(avg(case when U12M = 1 then VAL_BONUS else NULL end),2) as VL_SUM_U12M_VAL_BONUS,
    round(count(case when U12M = 1 then VAL_BONUS else NULL end),2) as VL_COUNT_U12M_VAL_BONUS,
    round(avg(case when U12M = 1 then VAL_BONUS else NULL end),2) as VL_MED_U12M_VAL_BONUS   
    
    FROM df_temp_01
GROUP BY NUM_CPF
ORDER BY NUM_CPF
""")

df_temp_03.createOrReplaceTempView("df_temp_03")
df_temp_03.count()

In [0]:
display(df_temp_03.limit(10))

In [0]:
# Dataset Finância

df_temp_04 = spark.sql("""

SELECT
    NUM_CPF,
    -- Para 1 mês 
    round(sum(case when U1M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_SUM_U1M_VAL_CREDITO_INSERIDO,
    round(sum(case when U1M = 1 then VAL_BONUS else NULL end),2) as VL_SUM_U1M_VAL_BONUS,
    round(sum(case when U1M = 1 then VAL_REAL else NULL end),2) as VL_SUM_U1M_VAL_REAL,
    round(count(case when U1M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_COUNT_U1M_VAL_CREDITO_INSERIDO,
    round(count(case when U1M = 1 then VAL_BONUS else NULL end),2) as VL_COUNT_U1M_VAL_BONUS,
    round(count(case when U1M = 1 then VAL_REAL else NULL end),2) as VL_COUNT_U1M_VAL_REAL,
    round(avg(case when U1M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_MED_U1M_VAL_CREDITO_INSERIDO,
    round(avg(case when U1M = 1 then VAL_BONUS else NULL end),2) as VL_MED_U1M_VAL_BONUS,
    round(avg(case when U1M = 1 then VAL_REAL else NULL end),2) as VL_MED_U1M_VAL_REAL,

    -- Para 3 meses
    round(sum(case when U3M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_SUM_U3M_VAL_CREDITO_INSERIDO,
    round(sum(case when U3M = 1 then VAL_BONUS else NULL end),2) as VL_SUM_U3M_VAL_BONUS,
    round(sum(case when U3M = 1 then VAL_REAL else NULL end),2) as VL_SUM_U3M_VAL_REAL,
    round(count(case when U3M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_COUNT_U3M_VAL_CREDITO_INSERIDO,
    round(count(case when U3M = 1 then VAL_BONUS else NULL end),2) as VL_COUNT_U3M_VAL_BONUS,
    round(count(case when U3M = 1 then VAL_REAL else NULL end),2) as VL_COUNT_U3M_VAL_REAL,
    round(avg(case when U3M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_MED_U3M_VAL_CREDITO_INSERIDO,
    round(avg(case when U3M = 1 then VAL_BONUS else NULL end),2) as VL_MED_U3M_VAL_BONUS,    
    round(avg(case when U3M = 1 then VAL_REAL else NULL end),2) as VL_MED_U3M_VAL_REAL,

    
    -- Para 6 meses
    round(sum(case when U6M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_SUM_U6M_VAL_CREDITO_INSERIDO,
    round(sum(case when U6M = 1 then VAL_BONUS else NULL end),2) as VL_SUM_U6M_VAL_BONUS,
    round(sum(case when U6M = 1 then VAL_REAL else NULL end),2) as VL_SUM_U6M_VAL_REAL,
    round(count(case when U6M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_COUNT_U6M_VAL_CREDITO_INSERIDO,
    round(count(case when U6M = 1 then VAL_BONUS else NULL end),2) as VL_COUNT_U6M_VAL_BONUS,
    round(count(case when U6M = 1 then VAL_REAL else NULL end),2) as VL_COUNT_U6M_VAL_REAL,
    round(avg(case when U6M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_MED_U6M_VAL_CREDITO_INSERIDO,
    round(avg(case when U6M = 1 then VAL_BONUS else NULL end),2) as VL_MED_U6M_VAL_BONUS,
    round(avg(case when U6M = 1 then VAL_REAL else NULL end),2) as VL_MED_U6M_VAL_REAL,

    -- Para 9 meses
    round(sum(case when U9M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_SUM_U9M_VAL_CREDITO_INSERIDO,
    round(sum(case when U9M = 1 then VAL_BONUS else NULL end),2) as VL_SUM_U9M_VAL_BONUS,
    round(sum(case when U9M = 1 then VAL_REAL else NULL end),2) as VL_SUM_U9M_VAL_REAL,
    round(count(case when U9M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_COUNT_U9M_VAL_CREDITO_INSERIDO,
    round(count(case when U9M = 1 then VAL_BONUS else NULL end),2) as VL_COUNT_U9M_VAL_BONUS,
    round(count(case when U9M = 1 then VAL_REAL else NULL end),2) as VL_COUNT_U9M_VAL_REAL,
    round(avg(case when U9M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_MED_U9M_VAL_CREDITO_INSERIDO,
    round(avg(case when U9M = 1 then VAL_BONUS else NULL end),2) as VL_MED_U9M_VAL_BONUS,
    round(avg(case when U9M = 1 then VAL_REAL else NULL end),2) as VL_MED_U9M_VAL_REAL,

    -- Para 12 meses
    round(sum(case when U12M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_SUM_U12M_VAL_CREDITO_INSERIDO,
    round(sum(case when U12M = 1 then VAL_BONUS else NULL end),2) as VL_SUM_U12M_VAL_BONUS,
    round(sum(case when U12M = 1 then VAL_REAL else NULL end),2) as VL_SUM_U12M_VAL_REAL,
    round(count(case when U12M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_COUNT_U12M_VAL_CREDITO_INSERIDO,
    round(count(case when U12M = 1 then VAL_BONUS else NULL end),2) as VL_COUNT_U12M_VAL_BONUS,
    round(count(case when U12M = 1 then VAL_REAL else NULL end),2) as VL_COUNT_U12M_VAL_REAL,
    round(avg(case when U12M = 1 then VAL_CREDITO_INSERIDO else NULL end),2) as VL_MED_U12M_VAL_CREDITO_INSERIDO,
    round(avg(case when U12M = 1 then VAL_BONUS else NULL end),2) as VL_MED_U12M_VAL_BONUS,
    round(avg(case when U12M = 1 then VAL_REAL else NULL end),2) as VL_MED_U12M_VAL_REAL
    
    FROM df_temp_01
GROUP BY NUM_CPF
ORDER BY NUM_CPF
""")

df_temp_04.createOrReplaceTempView("df_temp_04")
df_temp_04.count()

In [0]:
display(df_temp_04.limit(10))

In [0]:
#Dataset sequencia negativa ? - melhorar nome 

df_temp_06 = spark.sql("""

SELECT
    NUM_CPF,
    VL_SUM_U1M_VAL_REAL,
    VL_SUM_U3M_VAL_REAL,
    VL_SUM_U6M_VAL_REAL,
    VL_SUM_U9M_VAL_REAL,
    VL_SUM_U12M_VAL_REAL
FROM
    df_temp_04
WHERE
    VL_SUM_U1M_VAL_REAL < 0 OR
    VL_SUM_U3M_VAL_REAL < 0 OR
    VL_SUM_U6M_VAL_REAL < 0 OR
    VL_SUM_U9M_VAL_REAL < 0 OR
    VL_SUM_U12M_VAL_REAL < 0 
ORDER BY NUM_CPF
""")

df_temp_06.createOrReplaceTempView("df_temp_06")
df_temp_06.count()

Pode verificar que pelo menos 136212 já esteve negativado nos ultimos 12 meses (202410 - 202310)

In [0]:
display(df_temp_06.limit(10))

In [0]:
#Dataset daquele que apenas receberam bonus

df_temp_07 = spark.sql("""

SELECT
    NUM_CPF,
    VL_SUM_U1M_VAL_BONUS,
    VL_SUM_U3M_VAL_BONUS,
    VL_SUM_U6M_VAL_BONUS,
    VL_SUM_U9M_VAL_BONUS,
    VL_SUM_U12M_VAL_BONUS
FROM df_temp_04
WHERE VL_SUM_U1M_VAL_CREDITO_INSERIDO = 0 AND VL_SUM_U3M_VAL_CREDITO_INSERIDO = 0 AND VL_SUM_U6M_VAL_CREDITO_INSERIDO = 0 AND VL_SUM_U9M_VAL_CREDITO_INSERIDO = 0 AND VL_SUM_U12M_VAL_CREDITO_INSERIDO = 0 AND VL_SUM_U1M_VAL_BONUS != 0 AND VL_SUM_U3M_VAL_BONUS != 0 AND VL_SUM_U6M_VAL_BONUS != 0 AND VL_SUM_U9M_VAL_BONUS != 0 AND VL_SUM_U12M_VAL_BONUS != 0
""")

df_temp_07.count()

In [0]:
display(df_temp_07.limit(10))